# APK 6725 - AI for Sport & Movement Sciences
## Assignment 8: Trees, Forests and Honest Evaluation

This assignment practices the ideas from lectures 04.02 to 04.04. You will classify **daily activities** from smartphone sensor data.

- **Part A (2.5 points):** decision trees, random forests, and a comparison with logistic regression and kNN.
- **Part B (2.5 points):** how honest is the test? Data leakage, and how many subjects you need.

Write your code in the cells provided and test it. Some cells train many models: they can take 1 to 2 minutes in Colab.

Total: 5 points.

## Run this first

Run the cell below, at the start of every session. It downloads the course files to your Google Drive and keeps them up to date. When you see "Setup complete", you are ready to work.

**Do not edit or save your work inside the `AIforMovementScience` folder.** Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/Assignments'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
DATA = REPO + '/lectures/data'        # the folder that holds the data files
print('Setup complete. You can now run the rest of the notebook.')

## How to submit

Before you submit, please make sure your notebook is organized and reproducible. Part of your grade is for this.

- Answer each question in its own section, using the headings already provided.
- Put your code in the code cells; when a question asks for a written answer, use the markdown answer cell.
- Delete any scratch or experimental cells that are not part of your answer.
- Download your notebook: **File > Download > Download .ipynb**, name it `YourName_Assignment_8.ipynb`, and upload it to Canvas.

---
# The data: Human Activity Recognition (UCI HAR)

Thirty volunteers (19 to 48 years old) did six activities while they wore a smartphone on the waist:

1. Walking
2. Walking upstairs
3. Walking downstairs
4. Sitting
5. Standing
6. Laying

The phone recorded the accelerometer and the gyroscope at 50 samples per second. The authors cut the signals into **windows** of 2.56 s (128 samples). Consecutive windows **overlap by 50%**: each window shares half of its samples with the next window.

For each window, the authors computed **561 features**: for example the mean, the standard deviation, the maximum and the energy of each signal, in the time and in the frequency domain, and angles between the phone and gravity. So each row of the data is one window, with 561 features and one activity label.

The authors split the data **by subject**: 21 subjects in the training set and 9 subjects in the test set.

More information: <https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones>

The data is in `lectures/data/UCI_HAR_Dataset`. Run the cell below to load it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HAR = f"{DATA}/UCI_HAR_Dataset"
activities = ['Walking', 'Upstairs', 'Downstairs', 'Sitting', 'Standing', 'Laying']

# some of the 561 feature names are repeated, so we add a number to make each name unique
names = pd.read_csv(f"{HAR}/features.txt", sep=r"\s+", header=None)[1]
feature_names = [f"{n} [{i}]" for i, n in enumerate(names, start=1)]

def load(part):
    X = pd.read_csv(f"{HAR}/{part}/X_{part}.txt", sep=r"\s+", header=None, names=feature_names)
    y = pd.read_csv(f"{HAR}/{part}/y_{part}.txt", header=None)[0].map(lambda k: activities[k - 1])
    subjects = pd.read_csv(f"{HAR}/{part}/subject_{part}.txt", header=None)[0]
    return X, y, subjects

X_train, y_train, subjects_train = load('train')
X_test, y_test, subjects_test = load('test')
print('Data loaded.')
X_train.iloc[:5, :6]

---
# Part A - Trees and forests (2.5 points)

## Question A1 (0.25 points) - Look at the data

1. Print the shape of `X_train` and of `X_test`.
2. Print the number of different subjects in the train set and in the test set. Check that no subject is in both sets (hint: `set(subjects_train) & set(subjects_test)`).
3. Print the number of windows of each activity in the train set (`y_train.value_counts()`).
4. A very simple "model" always predicts the most common activity of the train set. Compute its accuracy on the **test** set.

In the answer cell: how many features does each window have? About how many windows does one subject give? What accuracy must a useful model beat?

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

## Question A2 (0.75 points) - One decision tree

Answer parts (a) and (b), each in its own cell.

### Part (a) - Train and test accuracy for each depth

For each `max_depth` in `[1, 2, 3, 4, 6, 8, 10, 15, 20]`, train a `DecisionTreeClassifier(max_depth=..., random_state=0)` on the train set. Store the accuracy on the train set and on the test set. Plot the two accuracies against the depth, on the same axes, with a legend.

**Note:** here we look at the test accuracy only to **see** the overfitting, as in lecture 04.02. To **select** a hyperparameter, use cross-validation on the train set (Question A3).

In the answer cell: what happens to the train accuracy and to the test accuracy when the depth increases? After which depth does the test accuracy stop improving? What does the gap between the two curves tell you?

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

### Part (b) - Read a small tree

Train a tree with `max_depth=2` and plot it:

```Python
plt.figure(figsize=(16, 6))
plot_tree(tree_2, feature_names=feature_names, class_names=tree_2.classes_, filled=True, fontsize=9)
plt.show()
```

In the answer cell: which feature is used at the first split (the top of the tree)? Which activities go to the left and which go to the right? Explain why this split makes sense for the body. (Hint: look at the `value` row of each box. The order of the classes is `tree_2.classes_`.)

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

## Question A3 (1 point) - A random forest, tuned the right way

Answer parts (a) and (b), each in its own cell.

### Part (a) - Tune with group-aware cross-validation

Use `GridSearchCV` to select the hyperparameters of a random forest. Use `GroupKFold`, with the subjects as groups, so that the windows of one subject are never in the training part and in the validation part of the same fold.

```Python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold

param_grid = {'max_depth': [5, 10, None], 'min_samples_leaf': [1, 5]}
search = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
                      param_grid, cv=GroupKFold(n_splits=3), scoring='accuracy')
search.fit(X_train, y_train, groups=subjects_train)    # this takes about 1 minute
```

Print the best hyperparameters and the best cross-validation accuracy.

In the answer cell: why do we use `GroupKFold` here, and not the normal `KFold`? Is the test set used in this step?

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

### Part (b) - Evaluate on the test set

Use the best model (`search.best_estimator_`). On the test set:

1. Print the accuracy.
2. Print the `classification_report`.
3. Plot the confusion matrix:
   ```Python
   from sklearn.metrics import classification_report, ConfusionMatrixDisplay
   ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=activities, xticks_rotation=45, cmap='Blues')
   ```

In the answer cell:
- Which two activities does the model confuse most? Why does this make sense for a phone on the waist?
- Look at the precision and the recall of `Sitting`. Say in words what each number means for this activity (lecture 04.02).

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

## Question A4 (0.5 points) - Compare with other models, and look inside the forest

Answer parts (a) and (b), each in its own cell.

### Part (a) - Logistic regression and kNN

Train these two models on the train set and print their test accuracy. Both use distances or coefficients, so they need scaled features.

```Python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

lr  = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
knn = Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=15))])
```

In the answer cell: rank the three models (random forest, logistic regression, kNN) by test accuracy. Is the most flexible model always the best? Why can a linear model work well with these 561 features?

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

### Part (b) - Feature importance

Make a horizontal bar plot of the 10 most important features of the random forest (`rf.feature_importances_`). Use the names in `feature_names`.

In the answer cell: what type of information do most of the top features measure? (Hint: in the feature names, `Gravity` is the slow part of the acceleration, caused by gravity; `Body` is the fast part, caused by the movement; `angle(...)` is an angle between the phone and gravity.) Why are these features useful to separate the activities?

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

---
# Part B - How honest is the test? (2.5 points)

In Part A, the authors split the data by subject. What happens if we split the windows at random? And how many subjects do we need for a good model?

## Question B1 (1 point) - Data leakage: split by window or by subject?

Put all 30 subjects together:

```Python
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
subjects_all = pd.concat([subjects_train, subjects_test], ignore_index=True)
```

Repeat this experiment for `seed` in `[0, 1, 2]`:

1. **Split by window:** `train_test_split(X_all, y_all, test_size=0.3, random_state=seed, stratify=y_all)`.
2. **Split by subject:** `GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)` with `groups=subjects_all`. (Lecture 04.03 shows how to get `train_idx` and `test_idx`.)
3. For each split, train `RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)` on the train part and compute the accuracy on the test part.

Print the six accuracies, and the mean accuracy of each type of split.

In the answer cell:
- Which split gives the higher accuracy?
- Give **two** reasons why. (Hint: remember how the windows were made, and think about the movement style of each person.)
- A clinic wants to use the model on **new** patients. Which number should you report to the clinic?
- In lecture 04.04, the split by row and the split by subject gave almost the same AUC for the PD data. Why is the difference much larger here?

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

## Question B2 (1 point) - How many subjects do you need?

Keep the official test set (9 subjects). Train the random forest with only **some** of the 21 train subjects, and see how the test accuracy changes. For each number of subjects, repeat 5 times with different subjects chosen at random:

```Python
rng = np.random.RandomState(0)
train_ids = subjects_train.unique()
n_list = [2, 4, 8, 14, 21]
mean_acc, std_acc = [], []
for n in n_list:
    accs = []
    for repeat in range(5):
        chosen = rng.choice(train_ids, n, replace=False)
        rows = subjects_train.isin(chosen)
        model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
        model.fit(X_train[rows], y_train[rows])
        accs.append(model.score(X_test, y_test))
    # complete the loop: store the mean and the standard deviation of accs
```

This takes about 1 minute. Then plot the mean test accuracy against the number of subjects, with the standard deviation as error bars (`plt.errorbar(n_list, mean_acc, yerr=std_acc, marker='o', capsize=4)`). Label the axes.

In the answer cell:
- How do the mean and the spread of the test accuracy change with the number of subjects?
- Why is the standard deviation zero for 21 subjects?
- Two subjects give about 700 windows. Is that a lot of data? Explain the difference between **many samples** and **many subjects**.
- Would a model trained on 4 subjects be a good proof of principle? Would it be a good final model?

In [ ]:
# Write your code here

**Your answer:**  *(double-click this cell to write your answer)*

## Question B3 (0.5 points) - Read a result critically

A research paper reports a model that detects freezing of gait in Parkinson's disease with **99% accuracy**. The authors recorded 5 patients with a sensor on the ankle, cut the signals into 2-second windows with 50% overlap, and split the **windows** at random into 80% train and 20% test.

In the answer cell:
- Give **two** problems with this evaluation. Use what you learned in B1, B2 and lecture 04.04.
- Describe how you would evaluate the model in a more honest way.
- Accuracy may not be the best metric here: freezing happens in only a small part of the time. Which metrics from lecture 04.02 would you report, and why?

**Your answer:**  *(double-click this cell to write your answer)*

---
## Extra (no points)

- In A4(a), try `n_neighbors` = 1, 5, 50 and 150 for kNN. Which value is best? Is the result the same as with the moons data in lecture 04.02?
- Train the random forest with only the features that have `angle` in their name (`[f for f in feature_names if 'angle' in f]`). How many features is this? What accuracy do you get?